# Importing Libraries

In [ ]:
import pandas as pd
import numpy
#!pip install imbalanced-learn
from nltk.corpus import stopwords
import re
import string
import seaborn as sns
from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from matplotlib import pyplot as plt

# Reading Dataset

In [ ]:

data = pd.read_csv('cyberbullying_tweets(1).csv')
data

# Data Analysis

In [ ]:
data.info()

### Checking Null Values

In [ ]:
data.isnull().sum()

In [ ]:
data['cyberbullying_type'].value_counts()

# Data Preprocessing

### Renaming Columns

In [ ]:
data = data.rename(columns={'tweet_text': 'text', 'cyberbullying_type': 'sentiment'})

### Encoding Columns

In [ ]:
data["sentiment_encoded"] = data['sentiment'].replace({"religion": 1, "age": 2, "ethnicity": 3, "gender": 4, "other_cyberbullying": 5,"not_cyberbullying": 6})

In [ ]:
data.sample(10)

In [ ]:
import nltk
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

### Fucntion to Convert text to lowercase, remove (/r, /n characters), URLs, non-utf characters, Numbers, punctuations,stopwords

In [ ]:
def strip_all_entities(text):
    text = text.replace('\r', '').replace('\n', ' ').lower()
    text = re.sub(r"(?:\@|https?\://)\S+", "", text)
    text = re.sub(r'[^\x00-\x7f]',r'', text)
    text = re.sub(r'(.)1+', r'1', text)
    text = re.sub('[0-9]+', '', text)
    stopchars= string.punctuation
    table = str.maketrans('', '', stopchars)
    text = text.translate(table)
    text = [word for word in text.split() if word not in stop_words]
    text = ' '.join(text)
    return text

### Function to remove contractions


In [ ]:
def decontract(text):
    text = re.sub(r"can\'t", "can not", text)
    text = re.sub(r"n\'t", " not", text)
    text = re.sub(r"\'re", " are", text)
    text = re.sub(r"\'s", " is", text)
    text = re.sub(r"\'d", " would", text)
    text = re.sub(r"\'ll", " will", text)
    text = re.sub(r"\'t", " not", text)
    text = re.sub(r"\'ve", " have", text)
    text = re.sub(r"\'m", " am", text)
    return text

In [ ]:
def preprocess(text):
    text = decontract(text)
    text = strip_all_entities(text)
    return text

In [ ]:
data

In [ ]:
data['cleaned_text'] = data['text'].apply(preprocess)
data.head()

In [ ]:
data["cleaned_text"].duplicated().sum()

In [ ]:
data.drop_duplicates("cleaned_text", inplace=True)

In [ ]:
data.sentiment.value_counts()

### Extracting Text Length

In [ ]:
text_len = []
for text in data.cleaned_text:
    tweet_len = len(text.split())
    text_len.append(tweet_len)

In [ ]:
data['text_len'] = text_len

In [ ]:
data

In [ ]:
plt.figure(figsize=(7,5))
ax = sns.countplot(x='text_len', data=data[data['text_len']<10], palette='mako')
plt.title('Count of tweets with less than 10 words', fontsize=20)
plt.yticks([])
ax.bar_label(ax.containers[0])
plt.ylabel('count')
plt.xlabel('')
plt.show()

In [ ]:
data.sort_values(by=['text_len'], ascending=False)
data

### Removing Tweets which have less than 3 words and greater than 100 words

In [ ]:
data = data[data['text_len'] > 3]
data = data[data['text_len'] < 100]

In [ ]:
data

### Building Wordcloud for Every Class

In [ ]:
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator
from sklearn.preprocessing import LabelEncoder

lenc = LabelEncoder()
data.sentiment = lenc.fit_transform(data.sentiment)

for c in range(len(lenc.classes_)):
    string = ""
    for i in data[data.sentiment == c].cleaned_text.values:
        string = string + " " + i.strip()

    #custom_mask = np.array(Image.open('../input/twitter-pic/twitter.png'))
    wordcloud = WordCloud(width = 800, height = 800,
                background_color ='white',
                min_font_size = 10).generate(string)

    # plot the WordCloud image
    plt.figure(figsize = (8, 8), facecolor = None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad = 0)
    plt.title(lenc.classes_[c])
    plt.show()
    del string

# Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.pipeline import Pipeline


tfidf = TfidfTransformer()
clf = CountVectorizer()

X_cv =  clf.fit_transform(data['cleaned_text'])

tf_transformer = TfidfTransformer(use_idf=True).fit(X_cv)
X_tf = tf_transformer.transform(X_cv)

In [ ]:
X_tf

### Train Test Split Dataset

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_tf, data['sentiment'], test_size=0.20, stratify=data['sentiment'], random_state=42)

In [ ]:
y_train.value_counts()

### Oversampling using SMOTE will be used to balance the train dataset as this algorithm helps to overcome the overfitting problem posed by random oversampling

In [ ]:
from imblearn.over_sampling import SMOTE
vc = y_train.value_counts()
while (vc[0] != vc[4]) or (vc[0] !=  vc[2]) or (vc[0] !=  vc[3]) or (vc[0] !=  vc[1]):
    smote = SMOTE(sampling_strategy='minority')
    X_train, y_train = smote.fit_resample(X_train, y_train)
    vc = y_train.value_counts()

y_train.value_counts()

# Model Building ( Random Forest Classifier )

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_clf = RandomForestClassifier()
rf_clf.fit(X_train, y_train)

# Evaluation

In [ ]:
from sklearn.metrics import classification_report
sentiments = ["religion","age","gender","ethnicity","not bullying",'other_cyberbullying']

rf_pred = rf_clf.predict(X_test)
print('Classification Report for Random Forest:\n',classification_report(y_test, rf_pred, target_names=sentiments))

In [ ]:
from sklearn.metrics import confusion_matrix
def print_confusion_matrix(confusion_matrix, class_names, figsize = (10,7), fontsize=14):
    df_cm = pd.DataFrame(
        confusion_matrix, index=class_names, columns=class_names,
    )
    fig = plt.figure(figsize=figsize)
    try:
        heatmap = sns.heatmap(df_cm, annot=True, fmt="d")
    except ValueError:
        raise ValueError("Confusion matrix values must be integers.")
    heatmap.yaxis.set_ticklabels(heatmap.yaxis.get_ticklabels(), rotation=0, ha='right', fontsize=fontsize)
    heatmap.xaxis.set_ticklabels(heatmap.xaxis.get_ticklabels(), rotation=45, ha='right', fontsize=fontsize)
    plt.ylabel('Truth')
    plt.xlabel('Prediction')

# Testing with Truth and Predicted Classes

In [ ]:
from sklearn import metrics
cm = metrics.confusion_matrix(y_test,rf_pred)
print_confusion_matrix(cm,sentiments)

# Cross Validation Accuracy

In [ ]:
from sklearn.model_selection import cross_val_score
RF_cv_score = cross_val_score(rf_clf,X_train, y_train, cv=3)
print('Cross validation score (Random Forest Classifier):', RF_cv_score.mean())